# Folivox on Kaggle - Web Studio (tunneled) + CLI fallback
Repo: https://github.com/vatavale/folivox - book to chaptered .m4b/.mp3 via CosyVoice 3.
Settings: Accelerator GPU T4 x2, Internet ON.


In [ ]:
# Cell 0 - Environment checks (GPU, Python, disk, net)
# NOTE: Folivox upsteam pins Python <3.13, but Kaggle now ships 3.13.
# We warn (not abort) and let Cell 2 try install / fall back to conda 3.11.
import shutil, subprocess, sys
print('python:', sys.version.split()[0])
OK_PY = [(3,10),(3,11),(3,12)]
if tuple(sys.version_info[:2]) not in OK_PY:
    print(f"WARNING: Python {sys.version.split()[0]} is outside Folivox supported 3.10-3.12.")
    print('Continuing anyway: Cell 2 tries native install, else uses conda python=3.11.')
else:
    print('Python version supported.')
try:
    out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
        capture_output=True, text=True, timeout=20).stdout
    print(out)
except Exception as e:
    print('WARNING no nvidia-smi:', e)
    print('Enable GPU: Settings -> Accelerator -> GPU T4 x2')
du = shutil.disk_usage('/kaggle/working')
print(f'/kaggle/working free: {du.free/1e9:.1f} GB (need ~10 GB)')
import urllib.request
for u in ['https://pypi.org/simple/','https://huggingface.co']:
    try:
        urllib.request.urlopen(u, timeout=10); print('OK net:', u)
    except Exception as e:
        print('NO NET:', u, e, '-> turn Internet ON')


In [ ]:
# Cell 1 - Writable paths (everything under /kaggle/working persists as output)
import os
from pathlib import Path
W = Path('/kaggle/working'); W.mkdir(exist_ok=True)
for d in ['models','books','out','folivox_data','folivox_config','hf_cache','torch_home']:
    (W/d).mkdir(parents=True, exist_ok=True)
os.environ.update({
    'FOLIVOX_MODEL_DIR': str(W/'models'/'Fun-CosyVoice3-0.5B'),
    'HF_HOME': str(W/'hf_cache'),
    'HUGGINGFACE_HUB_CACHE': str(W/'hf_cache'),
    'TORCH_HOME': str(W/'torch_home'),
    'XDG_DATA_HOME': str(W/'folivox_data'),
    'XDG_CONFIG_HOME': str(W/'folivox_config'),
    'FOLIVOX_CONFIG': str(W/'folivox_config'/'config.toml'),
})
print('FOLIVOX_MODEL_DIR =', os.environ['FOLIVOX_MODEL_DIR'])
!which ffmpeg || (apt-get update -qq && apt-get install -y -qq ffmpeg)
!ffmpeg -version 2>&1 | head -n 1


In [ ]:
# Cell 2 - Install Folivox (tries native python first, falls back to conda 3.11)
import sys, subprocess
print('installer python:', sys.version.split()[0])
!pip install -q --upgrade pip
# Path A: native install (works if Kaggle python is 3.10-3.12, may work on 3.13)
!pip install -q "folivox[all] @ git+https://github.com/vatavale/folivox.git" && echo NATIVE_INSTALL_OK || echo NATIVE_INSTALL_FAILED
!folivox --version 2>&1 | head -n 2


In [ ]:
# Cell 2b - FALLBACK if native install failed (Kaggle has no conda; use uv python 3.11).
# SKIP this cell if Cell 2 printed NATIVE_INSTALL_OK.
!curl -LsSf https://astral.sh/uv/install.sh | sh 2>&1 | tail -n 2
!ls -la $HOME/.local/bin/uv $HOME/.cargo/bin/uv 2>&1; export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"; which uv; uv --version
!export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"; uv python install 3.11 2>&1 | tail -n 5
!export PATH="$HOME/.local/bin:$HOME/.cargo/bin:$PATH"; uv venv /kaggle/working/folo --python 3.11 2>&1 | tail -n 5; ls -la /kaggle/working/folo/bin/ 2>&1 | head -n 10
!/kaggle/working/folo/bin/pip install --upgrade pip 2>&1 | tail -n 2
# CUDA torch first (cu121, py3.11 wheels), then folivox without deps-games:
!/kaggle/working/folo/bin/pip install "torch>=2.3,<2.6" "torchaudio>=2.3,<2.6" --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -n 5
!/kaggle/working/folo/bin/python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"
# Full (untruncated) folivox install log -> file + tail:
!/kaggle/working/folo/bin/pip install "folivox[all] @ git+https://github.com/vatavale/folivox.git" 2>&1 | tee /kaggle/working/folo_install.log | tail -n 30
!/kaggle/working/folo/bin/folivox --version && echo FOLO_INSTALL_OK || (echo FOLO_INSTALL_FAILED; echo '--- full log ---'; cat /kaggle/working/folo_install.log)


In [ ]:
# Cell 3 - Download models (~6 GB voice; +1.6 GB QA unless skipped).
# Uses FOLO env binary if Cell 2b ran, else native folivox.
!test -x /kaggle/working/folo/bin/folivox && export FOLO_BIN=/kaggle/working/folo/bin/folivox || export FOLO_BIN=folivox; echo "using: $FOLO_BIN"; $FOLO_BIN setup --skip-asr
# full QA models (1.6 GB extra): run "$FOLO_BIN setup" without --skip-asr


In [ ]:
# Cell 4 - Upload book + voice (or copy from /kaggle/input)
import glob
!ls -lh /kaggle/input 2>&1 | head -n 20 || true
!ls -lh /kaggle/working/books 2>&1 || mkdir -p /kaggle/working/books
print('Upload via paperclip -> Upload to /kaggle/working/books: book.fb2/epub/pdf + voice.wav + voice.txt')


In [ ]:
# Cell 4b - Pick files + inspect book + list voices (EDIT voice id as needed)
!test -x /kaggle/working/folo/bin/folivox && export FOLO_BIN=/kaggle/working/folo/bin/folivox || export FOLO_BIN=folivox; echo "using: $FOLO_BIN"
import glob
books = sorted(sum([glob.glob(f'/kaggle/working/**/*.{e}', recursive=True) for e in ['fb2','epub','pdf','docx','txt','zip']], []))
voices = sorted(sum([glob.glob(f'/kaggle/working/**/*.{e}', recursive=True) for e in ['wav','flac']], []))
print('BOOKS:', books or '- upload to /kaggle/working/books')
print('VOICES:', voices or '- upload 6-15 s clean WAV, or use bundled id')
BOOK = books[0] if books else '/kaggle/working/books/book.fb2'
print('BOOK =', repr(BOOK))
%env BOOK={BOOK}
!${FOLO_BIN:-folivox} inspect "$BOOK" --depth 2 || true
!${FOLO_BIN:-folivox} voices | head -n 40 || true
# register your own voice (needs wav + txt transcript):
# !${FOLO_BIN:-folivox} voices add myvoice /kaggle/working/books/myvoice.wav --text /kaggle/working/books/myvoice.txt --gender f
# !${FOLO_BIN:-folivox} bench --devices cuda --lang en || true


In [ ]:
# Cell 5 - Smoke test: preview first 500 chars (validates GPU + voice)
!test -x /kaggle/working/folo/bin/folivox && export FOLO_BIN=/kaggle/working/folo/bin/folivox || export FOLO_BIN=folivox; echo "using: $FOLO_BIN"
!$FOLO_BIN render "$BOOK" --voice en-kalynda --preview 500 --no-qa -o /kaggle/working/out/preview.m4b --work-dir /kaggle/working/out/preview.folivox || true
!ls -lh /kaggle/working/out/ 2>&1


In [ ]:
# Cell 6a - Write kaggle_patch.py inline (push uploads only the .ipynb, not sibling files).
%%writefile /kaggle/working/kaggle_patch.py
"""Kaggle shim for Folivox Web Studio (folivox ui).

Why this exists
---------------
`folivox ui` is designed for a *local* PC:

* binds to 127.0.0.1:8765
* refuses non-loopback Host / Origin headers (DNS-rebinding guard)
* auto-opens a desktop browser

On Kaggle none of that works: the kernel runs in a container and your
browser is elsewhere. The notebook therefore:

1. starts uvicorn on 0.0.0.0:8765 in a background thread,
2. exposes it with a tunnel (cloudflared),
3. wraps the Starlette app so the loopback guard sees "127.0.0.1"
   while the real tunnel Host is preserved for routing.

Security: the ?token= check is NOT disabled. Anyone opening the tunnel
URL still needs the token printed by the notebook. Do not share that URL.
Stop the notebook / tunnel when done.

Usage inside Kaggle (see kaggle_folivox.ipynb Cell 6):

    import kaggle_patch
    info = kaggle_patch.launch(port=8765)
    print(info["local_url"], info["token"])
    # then start cloudflared in another cell and open <tunnel_url>/?token=<token>
"""

from __future__ import annotations

import threading
from pathlib import Path


def get_token() -> str:
    """Reuse the same persistent token folivox ui uses."""
    from folivox.ui import ui_token

    return ui_token()


def _normalise_scope_for_guard(scope: dict) -> None:
    """Rewrite Host/Origin seen by the inner app to loopback.

    The tunnel (e.g. https://xyz.trycloudflare.com) forwards with
    Host: xyz.trycloudflare.com which folivox's rebinding guard would
    reject. We spoof only what the *inner* app sees so its check passes.
    Token auth still runs on the query string / cookie, untouched.
    """
    try:
        headers = scope.get("headers")
        if not headers:
            return
        new_headers = []
        for k, v in headers:
            lk = k.lower()
            if lk == b"host":
                new_headers.append((k, b"127.0.0.1:8765"))
            elif lk in (b"origin", b"referer"):
                # point origin back at loopback so an Origin check passes;
                # the browser still sends the real token cookie / query.
                try:
                    from urllib.parse import urlparse, urlunparse

                    u = urlparse(v.decode("latin-1"))
                    u = u._replace(netloc="127.0.0.1:8765")
                    new_headers.append((k, urlunparse(u).encode("latin-1")))
                except Exception:
                    new_headers.append((k, v))
            else:
                new_headers.append((k, v))
        scope["headers"] = new_headers
        # Some guards also check scope["server"]; normalise it too.
        if scope.get("type") in ("http", "websocket"):
            scope["server"] = ("127.0.0.1", scope.get("server", (None, None))[1] or 8765)
            scope["client"] = scope.get("client") or ("127.0.0.1", 0)
    except Exception:
        pass


def create_kaggle_app(token: str, port: int = 8765):
    """Call original create_app() then wrap with guard-normalising middleware."""
    from folivox.ui.app import create_app

    inner = create_app(token=token, port=port)

    async def kaggle_guard_wrapper(scope, receive, send):
        if scope["type"] in ("http", "websocket"):
            _normalise_scope_for_guard(scope)
        await inner(scope, receive, send)

    return kaggle_guard_wrapper


def launch(port: int = 8765, log_level: str = "warning") -> dict:
    """Start the Studio server in a background thread on 0.0.0.0.

    Returns {"port","token","local_url","thread"}.
    Open it via a tunnel: <tunnel_url>/?token=<token>
    """
    import uvicorn

    token = get_token()
    app = create_kaggle_app(token=token, port=port)

    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=port,
        log_level=log_level,
        proxy_headers=True,
        forwarded_allow_ips="*",
    )
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True, name="folivox-kaggle-ui")
    thread.start()
    return {
        "port": port,
        "token": token,
        "local_url": f"http://127.0.0.1:{port}/?token={token}",
        "thread": thread,
        "server": server,
    }


if __name__ == "__main__":
    info = launch()
    print("Folivox Kaggle Studio (patched) running on 0.0.0.0:%s" % info["port"])
    print("Local URL (inside kernel only):", info["local_url"])
    print("Via tunnel open: <your-tunnel-url>/?token=%s" % info["token"])
    info["thread"].join()
# (end of embedded patch; run this cell, then run Cell 6b below)


In [ ]:
# Cell 6b - Expose Studio with cloudflared; then open <tunnel>/?token=<token>
import subprocess, time, re, os
os.system('curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared && chmod +x /tmp/cloudflared && /tmp/cloudflared --version')
log = open('/tmp/cf.log','w')
subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://127.0.0.1:8765'], stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    txt = open('/tmp/cf.log').read()
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', txt)
    if m: url = m.group(0); break
print('tunnel:', url or 'not ready - wait 30 s and re-run; log: /tmp/cf.log')
tok = globals().get('info', {}).get('token', '<TOKEN from Cell 6>')
import pathlib
for p in [pathlib.Path('/root/.local/share/folivox/ui_token'), pathlib.Path.home()/'.local/share/folivox/ui_token']:
    if p.exists(): tok = p.read_text().strip(); break
if url: print(f'OPEN: {url}/?token={tok}')


In [ ]:
# Cell 7 - CLI fallback: full render (same pipeline as Studio). Resume-safe.
!test -x /kaggle/working/folo/bin/folivox && export FOLO_BIN=/kaggle/working/folo/bin/folivox || export FOLO_BIN=folivox; echo "using: $FOLO_BIN"
!$FOLO_BIN render "$BOOK" --voice en-kalynda -c 1-2 -o /kaggle/working/out/book_ch1-2.m4b --work-dir /kaggle/working/out/book.folivox || true
# whole book: !$FOLO_BIN render "$BOOK" --voice en-kalynda -o /kaggle/working/out/book.m4b --work-dir /kaggle/working/out/book.folivox
!ls -lh /kaggle/working/out/ 2>&1


In [ ]:
# Cell 8 - Download results + persist model cache
from IPython.display import FileLink
import glob
for f in sorted(glob.glob('/kaggle/working/out/*.m4b') + glob.glob('/kaggle/working/out/*.mp3')):
    print(f); display(FileLink(f))
print('To cache models: Save Version -> Save Output, or Datasets -> New -> upload models + hf_cache.')
print('Next session: attach that Dataset, set FOLIVOX_MODEL_DIR to it before Cell 3.')
!${FOLO_BIN:-folivox} doctor --bundle /kaggle/working/out 2>&1 | tail -n 20 || true
